# Purpose

to take in files from skok/stony ford produced by blastitall, and then make nice plots 

# Imports

In [1]:
import numpy as np
import pandas as pd
import os, csv, glob
import matplotlib.pyplot as plt

In [196]:
from Bio.Blast import NCBIWWW
from Bio import SeqIO 
from Bio.Blast import NCBIXML
from Bio import Phylo

In [3]:
from thefuzz import fuzz

# Functions

In [4]:
fld = 'Z:/dennise/20260106_subset'

In [92]:
es=[]
genera=[]
species=[]
files=[]
fileinfo=[]
seqnums=[]
for file in os.listdir(fld):
    seqnums.append(int(file.split('.tsv_')[1][:-4]))
    fileinfo.append(file.split('results_')[1].split('_merged')[0])
    df = pd.read_csv(os.path.join(fld,file))
    minexpect=np.min(df.expect)
    if not np.isnan(minexpect):
        subdf=df[df.expect==minexpect].copy()
        good_indices=[]
        if len(subdf)>1:
            # check if genera are all the same, then if species are all the same
            subspec=[spec.split(' ')[1] for spec in subdf.hit_definition]
            for val in np.unique(subspec):
                same_indices=[i for i, sp in enumerate(subspec) if sp == val]
                good_indices.append(same_indices[0])
            subdf=subdf.loc[good_indices]
        for idx in subdf.index:
            files.append(file)
            genera.append(subdf.hit_definition[idx].split(' ')[0])
            species.append(subdf.hit_definition[idx].split(' ')[1])
            es.append(minexpect)
newdf=pd.DataFrame()
newdf['file']=files
newdf['file_info']=fileinfo
newdf['seq_num']=seqnums
newdf['genus']=genera
newdf['species']=species
newdf['e_val']=es
newdf

,file,genus,species,e_val
0,results_28_F_merged.tsv_0.csv,Pseudochlorella,signiensis,2.327400e-32
1,results_28_F_merged.tsv_1.csv,Pseudochlorella,signiensis,9.896340e-31
2,results_28_F_merged.tsv_10.csv,Orchestia,gammarellus,5.305100e-34
3,results_28_F_merged.tsv_11.csv,Neotrombicula,inopinata,5.473510e-34
4,results_28_F_merged.tsv_12.csv,Diplosphaera,sundellii,3.235130e-24
5,results_28_F_merged.tsv_13.csv,Neotrombicula,inopinata,4.492930e-35
6,results_28_F_merged.tsv_14.csv,Pseudochlorella,signiensis,3.688020e-36
7,results_28_F_merged.tsv_15.csv,Orchestia,gammarellus,1.519940e-34
8,results_28_F_merged.tsv_2.csv,Orchestia,gammarellus,2.934170e-37
9,results_28_F_merged.tsv_3.csv,Neotrombicula,inopinata,1.910440e-33


In [ ]:
## next I need to figure out how to merge similar sequences using Ns

In [170]:
df = pd.read_csv('Z:/dennise/skok72_out/28_F_merged.tsv_info.csv')
df=df.drop(columns='Unnamed: 0')
df['groups']=0
df

,list_of_seqs,n_similar,groups
0,caatcattgctcgcattaccataaaaaaaatcattagaaaagcgtg...,913,0
1,caatcattgctngcattaccataaaaaaaatcattagaaaancgtg...,2035,0
2,aaccatttactntgccatagnttcttgagctggtgtaatngggacc...,31,0
3,tnngctttattngatntttggagcattttcaggagttcttggtact...,75,0
4,tactttatactttattctaggggcctgagctagtgtggtaggaacc...,1045,0
...,...,...,...
1741,aaccatttactatgncatagnttcttgagctggtgtaatagngncc...,22,0
1742,caattataattcgtattactntaaaaaaaattatcacaaaancatg...,22,0
1743,caattataataggtntaactntaaaaaaaattattacaaaancatg...,21,0
1744,aactttatattntatttttgggagttgngctngaatagtagnanct...,16,0


In [181]:
group_num=0
ns_in_seq=[]
for idx in df.index:
    seq = str(df.list_of_seqs[idx])
    seq_count=seq.count('n')
    ns_in_seq.append(seq_count)
    if df.groups[idx]==0:
        group_num+=1
        df.iloc[idx,'groups']=group_num
        print('idx',idx,group_num)
        for i in df.index[idx:]:
            if fuzz.ratio(seq,df.list_of_seqs[i])+np.max([seq_count,str(df.list_of_seqs[i]).count('n')])>97:
                df.iloc[i,'groups']=group_num
                print(i,group_num)
                #print(idx,group_num,fuzz.ratio(seq,df.list_of_seqs[i])+np.max([seq.count('n'),str(df.list_of_seqs[i]).count('n')]))
df['ns_in_seq']=ns_in_seq


In [195]:
seqs=[]
n_similar=[]
files_grouped=[]
for group in np.unique(df.groups):
    subdf=df[df.groups==group].copy()
    if len(subdf[subdf.ns_in_seq==np.min(subdf.ns_in_seq)])==1:
        keep_index = subdf.index[subdf.ns_in_seq==np.min(subdf.ns_in_seq)]
    else:
        keep_index=subdf.index[subdf.ns_in_seq==np.min(subdf.ns_in_seq)][0]
    seqs.append(str(subdf.list_of_seqs[subdf.ns_in_seq==np.min(subdf.ns_in_seq)].values[0]))
    n_similar.append(np.sum([val for val in subdf.n_similar]))
    files_grouped.append(list(subdf.index))
dfnew=pd.DataFrame()
dfnew['seqs']=seqs
dfnew['n_similar']=n_similar
dfnew['files_grouped']=files_grouped
dfnew

,seqs,n_similar,files_grouped
0,caatcattgctcgcattaccataaaaaaaatcattagaaaagcgtg...,4714,"[0, 1, 108, 119, 158, 180, 193, 254, 273, 283,..."
1,aaccatttactntgccatagcttcttgagctggtgtaatngggacc...,1388,"[2, 5, 14, 57, 58, 82, 91, 787, 1061, 1433]"
2,tnngctttattngatntttggagcattttcaggagttcttggtact...,75,[3]
3,tactttatactttattctaggggcctgagctagtgtggtaggaacc...,4069,"[4, 6, 8, 44, 45, 48, 54, 84, 89, 102, 103, 10..."
4,aactttatactttattttcggtgcgtgagcaggaatagtagnaaca...,2112,"[7, 17, 51, 100, 138, 154, 183, 197, 295, 689,..."
...,...,...,...
112,tncgctttatttattattcgccgcantttccggagttcttgntact...,46,"[1656, 1665]"
113,caataagaatangcattaccataaagaaaatcattaaaaaancatg...,17,[1673]
114,tactttatactntattctaggggcctgagctagtgtggtaggancc...,286,"[1711, 1713, 1715, 1716, 1722, 1723, 1724, 172..."
115,cgattaaagctngcataaccataaagaatatcattaatatancatg...,17,[1721]


## phylo

In [203]:
# using https://biopython.org/wiki/Phylo
trees = Phylo.read('/Users/dennise/Downloads/test.xml','phyloxml')
trees

ValueError: There are no trees in this file.

## old stuff

In [29]:
fld=hifld
col_nms=['file_nm','sci_nm','count']
df=pd.DataFrame(columns=col_nms)
df_seqdata=pd.DataFrame(columns=['file_nm','nuc_seq','count'])
anm_list=[]
for file_nm in [file for file in os.listdir(fld)]:
    print(file_nm)
    anm = file_nm[:-4]
    file = pd.read_table(os.path.join(fld,file_nm))
    unique_scinames = np.unique(file.SCIENTIFIC_NAME.dropna())
    list_of_vals=[]
    if len(unique_scinames)>0:
        for sci_nm in unique_scinames:
                list_of_vals.append([file_nm,sci_nm,len(file[file.SCIENTIFIC_NAME==sci_nm])])
                anm_list.append(anm)
    df2 = pd.DataFrame(list_of_vals, columns=col_nms+['anm_list'])
    df=pd.concat([df,df2],ignore_index=True)

s12_134_F.csv
s12_148_F.csv
s12_170_F.csv
s12_176_F.csv
s12_193_F.csv
s12_195_F.csv
s12_204_F.csv


In [ ]:
np.unique(file.)

array([nan])

In [36]:
file.columns

Index(['BEST_MATCH_IDS', 'ID', 'COUNT', 'DEFINITION', 'QUALITY',
       'BEST_MATCH_TAXIDS', 'NUC_SEQ', 'TAXID', 'BEST_IDENTITY', 'ID_STATUS',
       'SCIENTIFIC_NAME'],
      dtype='object')

In [6]:
df.to_csv('/home/dennislab2/Desktop/20240215_firstpass.csv')

In [94]:
df.to_csv('../../../uniplant_r_summary.csv')

In [10]:
obi_out_fld = '/home/dennislab2/Desktop/seqout5/'
#ncbi_out_fld = '/home/dennislab2/Desktop/ncbi_outputs/'

In [45]:
obi_file_paths = [os.path.join(obi_out_fld,file) for file in os.listdir(obi_out_fld) if '.csv' in file]
#obi_file_paths
#ncbi_file_paths = [os.path.join(ncbi_out_fld,file) for file in os.listdir(ncbi_out_fld) if '.csv' in file]
#ncbi_file_paths

In [263]:
s12_before_files=[]
s12_after_files=[]
for path in obi_file_paths:
    if "s12_216" in path:
        s12_before_files.append(path)
    elif "s12_255" in path:
        s12_after_files.append(path)

In [264]:
uniquebefore=[]
for file in s12_before_files:
    tempdf=pd.read_csv(file)
    uniques=np.unique(tempdf.hit_definition[0:10])
    for unique in uniques:
        if unique not in uniquebefore:
            uniquebefore.append(unique)
uniquebefore.sort()
for val in uniquebefore:
    if val not in notbugs:
        if ("ncultured" not in val) and ("16S" not in val) and ("strain" not in val) and ("ribosom" not in val) and ("MAG: " not in val) and ("musculus" not in val):
            print(val)

In [265]:
uniqueafter=[]
for file in s12_after_files:
    tempdf=pd.read_csv(file)
    uniques=np.unique(tempdf.hit_definition[0:10])
    for unique in uniques:
        if unique not in uniqueafter:
            uniqueafter.append(unique)
uniqueafter.sort()
for val in uniqueafter:
    if val not in notbugs:
        if ("ncultured" not in val) and ("16S" not in val) and ("strain" not in val) and ("ribosom" not in val) and ("MAG: " not in val) and ("musculus" not in val):
            print(val)

Alectoris rufa genome assembly, chromosome: 29
Bacteroides phage BF1-TP2, complete genome
Bifidobacterium longum subsp. longum JDM301, complete genome
Bifidobacterium scardovii JCM 12489 = DSM 13734 DNA, complete genome
Dichanthelium sp. BIOUG24049-A06 internal transcribed spacer 2, partial sequence
Didymella exigua CBS 183.55 uncharacterized protein (M421DRAFT_423552), mRNA
Erysipelotrichaceae bacterium ASTB_g chromosome, complete genome
Hapalemur griseus griseus mitochondrion partial 12S rRNA gene, isolate 10
Macroventuria anomochaeta uncharacterized protein (BU25DRAFT_413286), mRNA


##### 

# Use case

# Procedure

In [29]:
i=0
ncbi_file_name = ncbi_file_paths[i]
ncbi_file = pd.read_csv(ncbi_file_name).drop(columns=['Unnamed: 0'])
read_num = int(ncbi_file_name.split('_')[-1][:-4])
print(read_num)
ncbi_file.head()


27


,hit_definition,hit_accession,subject,identities,expect
0,"Mus musculus domesticus mitochondrial DNA, com...",AB092593,GACGGGCGGTGTGTGCGTACTTCATTGCTCAATTCAATTAAGCTCT...,140,1.248650e-58
1,"Mus musculus domesticus mitochondrial DNA, com...",AB092592,GACGGGCGGTGTGTGCGTACTTCATTGCTCAATTCAATTAAGCTCT...,140,1.248650e-58
2,"Mus musculus genome assembly, organelle: mitoc...",OX439034,GACGGGCGGTGTGTGCGTACTTCATTGCTCAATTCAATTAAGCTCT...,140,1.248650e-58
3,"Mus musculus genome assembly, organelle: mitoc...",OX390165,GACGGGCGGTGTGTGCGTACTTCATTGCTCAATTCAATTAAGCTCT...,140,1.248650e-58
4,"Mus musculus genome assembly, organelle: mitoc...",OX389814,GACGGGCGGTGTGTGCGTACTTCATTGCTCAATTCAATTAAGCTCT...,140,1.248650e-58


In [27]:
ncbi_file_name

'/home/dennislab2/Desktop/ncbi_outputs/results_plate1_12s_21_1_27.csv'

In [35]:
obi_file = pd.read_table(os.path.join(obi_out_fld,ncbi_file_name.split('results_')[1][:-7]+"_out.csv"))
seq=obi_file.NUC_SEQ[obi_file.index==read_num].values

array(['agagcgacgggcgatgtgtgcgtacttcattgctcaattcaattaagctctctattcttaatttactactaaatcctccttagtcctttagtttcataaagggtatagtaatgttcttttataagaaaatgtagcccatttcttcccatt'],
      dtype=object)

## want 
mouse | location | timepoint | seq_num | count | best_hit | identities

In [57]:
meta21

,plate,plate_letter,plate_number,sample_num,strain,left_eartag,right_eartag,location,wedge,infected,weeks,weight_change,block
0,1,A,1,1,C57,401,89.0,SF,3.0,0.0,2.0,5.4,1.0
1,1,B,1,2,C57,416,7024.0,SF,3.0,0.0,2.0,6.6,1.0
2,1,C,1,3,129,NaN,7010.0,SF,4.0,0.0,2.0,5.6,1.0
3,1,D,1,4,PWK,454,726.0,Lab,NaN,0.0,2.0,6.1,1.0
4,1,F,1,5,C57,413,81.0,SF,3.0,1.0,5.0,2.7,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,2,G,6,64,C57,286,25.0,Lab,NaN,1.0,11.5,5.0,2.0
128,2,G,12,68,129,447,7053.0,Lab,NaN,1.0,5.5,5.0,1.0
129,2,H,2,69,C57,285,24.0,Lab,NaN,1.0,7.3,5.0,2.0
130,2,H,9,73,C57,293,26.0,Lab,NaN,0.0,4.8,5.0,2.0


In [68]:
split_name = ncbi_file_name.split('_')
plate = int(split_name[2][-1])
sample_val = int(split_name[4])
f_or_r= int(split_name[5])

In [87]:
meta21[(meta21.sample_num==29) & (meta21.plate==1)]

,plate,plate_letter,plate_number,sample_num,strain,left_eartag,right_eartag,location,wedge,infected,weeks,weight_change,block
28,1,A,7,29,C57,408,77.0,SF,3.0,0.0,2.0,8.3,1.0


In [88]:
# todo - link plate and sample number from file to metadata, and add the plants and inverts in a smart way with counts. 

## make a combined metadata dataframe

In [41]:
# import files
df_sf=pd.read_csv('/home/dennislab2/Desktop/2021_SF_METADATA.csv')
df_seq=pd.read_csv('/home/dennislab2/Desktop/2021_SEQ_METADATA.csv')

In [42]:
df_sf

,plate,plate_letter,plate_number,strain,left_eartag,right_eartag,location,wedge,infected,weeks,weight_change,block
0,1,A,1,C57,401,89.0,SF,3.0,0.0,2.0,5.4,1.0
1,1,A,2,C57,402,7050.0,SF,3.0,0.0,2.0,4.3,1.0
2,1,A,4,C57,405,7021.0,Lab,NaN,0.0,2.0,1.8,1.0
3,1,A,5,C57,406,7030.0,SF,3.0,0.0,2.0,1.7,1.0
4,1,A,6,C57,407,7023.0,SF,3.0,0.0,2.0,3.8,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
235,3,H,8,129,105,19.0,Lab,NaN,0.0,9.4,5.0,2.0
236,3,H,9,C57,297,27.0,SF,NaN,1.0,1.8,5.0,2.0
237,3,H,10,C57,298,27.0,SF,NaN,0.0,1.6,5.0,2.0
238,3,H,11,C57,299,27.0,Lab,NaN,1.0,6.7,5.0,2.0


In [43]:
df_seq

,plate,plate_letter,plate_number,sample_num
0,1,A,1,1
1,1,B,1,2
2,1,C,1,3
3,1,D,1,4
4,1,F,1,5
...,...,...,...,...
129,2,H,5,71
130,2,H,7,72
131,2,H,9,73
132,2,H,10,74


In [56]:
meta21=pd.merge(df_seq,df_sf,on=["plate","plate_letter","plate_number"])
meta21.to_csv('/home/dennislab2/Desktop/2021_metadata.csv')

'A'